In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'experiments':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.metrics import f1_score

from src.preprocessing import load_data, clean_data, split_data, numeric_cols, categoric_cols
from src.discretization import continuous_cols, methods, apply_edges
from src.models import models, hyperparameters

In [ ]:
REPRESENTATION = 'original'  # TEMPORARY
N_BINS = 3
REPORT_TEST = False

# None is all 27 configs, 1 is a quick trial
LIMIT = None

clean = pd.read_csv(ROOT / 'data' / 'preprocessed' / 'adult_cleaned.csv')
X = train.drop(columns='income')
y = train['income']
X_test = test.drop(columns='income')
y_test = test['income']

In [ ]:
def holdout(seed):
    return train_test_split(np.arange(len(X)), test_size=0.2, random_state=seed, stratify=y)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

splits = {
    'holdout': [holdout(42)],
    'repeated': [holdout(seed) for seed in [42, 0, 27]],
    'cv': list(cv.split(X, y))
}

In [ ]:
def prepare_features(train_idx, val_idx, name):
    train = X.iloc[train_idx].copy()
    val = X.iloc[val_idx].copy()
    frames = [train, val]
    if REPORT_TEST:
        frames.append(X_test.copy())

    if REPRESENTATION != "original":
        for col in continuous_cols:
            edges = methods[REPRESENTATION](train[col], N_BINS)
            for frame in frames:
                frame[col] = apply_edges(frame[col], edges)

    encoder = ColumnTransformer([
        ("num", StandardScaler() if name == "kNN" else "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         [col for col in categoric_cols if col != "income"])
    ])

    encoder.fit(train)
    return [encoder.transform(frame) for frame in frames]

In [ ]:
def run_classifier(name):
    configs = list(ParameterGrid(hyperparameters[name]))[:LIMIT]

    table = pd.DataFrame([
        {
            'ID': i,
            **{k: 'None' if v is None else v for k, v in params.items()}
        }
        for i, params in enumerate(configs, start=1)
    ])

    metrics = ['train', 'val', 'test'] if REPORT_TEST else ['train', 'val']

    for strategy, partitions in splits.items():
        run_scores = []

        for train_idx, val_idx in partitions:
            matrices = prepare_features(train_idx, val_idx, name)
            labels = [y.iloc[train_idx], y.iloc[val_idx]]
            if REPORT_TEST:
                labels.append(y_test)

            scores = []
            for param in configs:
                model = clone(models[name]).set_params(**params)
                if name != 'kNN':
                    model.set_params(random_state=42)
                model.fit(matrices[0], labels[0])

                scores.append([
                    f1_score(
                        target,
                        model.predict(matrix),
                        pos_label='>50K',
                        zero_division=0
                    )
                    for target, matrix in zip(labels, matrices)
                ])

            run_scores.append(scores)


        # Averages
        averages = np.mean(run_scores, axis=0)
        for j, metric = enumerate(metrics):
            table[f'{strategy}_{metric}_F1'] = averages[:, j]

        print(name, strategy, 'complete')

    return table

In [ ]:
dt_results = run_classifier('Decision_Tree')
display(dt_results)

In [ ]:
knn_results = run_classifier('kNN')
display(knn_results)

In [ ]:
rf_results = run_classifier('Random_Forest')
display(rf_results)

In [ ]:
results = {"Decision_Tree": dt_results, "kNN": knn_results, "Random_Forest": rf_results}
criteria = ["holdout_val_F1", "repeated_val_F1", "cv_val_F1", "holdout_train_F1"]

for name, table in results.items():
    print(name)
    for criterion in criteria:
        best = table[table[criterion] == table[criterion].max()]
        print(criterion, "IDs:", best["ID"].tolist(), "F1:", round(best[criterion].iloc[0], 4))

    f1_cols = [col for col in table if col.endswith("_F1")]
    display(table.style.format(precision=4, subset=f1_cols)
            .highlight_max(subset=f1_cols, props="font-weight: bold;")
            .set_caption(f"{name}: model selection F1-scores"))

In [ ]:
if LIMIT is None and REPORT_TEST:
    output = ROOT / "results" / "tables" / "RQ1"
    output.mkdir(parents=True, exist_ok=True)
    for name, table in results.items():
        table.to_csv(output / f"{name}_results.csv", index=False)